# Subtype models: BERT optimized for F1-score

Trains BERT classifiers for the stigma subtypes using F1-based selection.

This notebook accompanies [Stigmatizing Language in Gender-Expansive Patient Records: Corpus Development, Disparity Analysis, and Natural Language Processing-Based Detection Study](https://www.jmir.org/2026/1/e91089).

## Data and execution requirements

- Clinical note text and MIMIC identifiers are not included in this repository.
- Run this notebook only in an environment authorized to access MIMIC-IV and the credentialed annotation release.
- Set `GEP_DATA_DIR`, `GEP_MODEL_DIR`, `GEP_RESULTS_DIR`, and `GEP_FIGURES_DIR` as needed. By default, repository-local directories are used.
- The notebook outputs and execution counters have been removed from the public version.

**Selection protocol.** For each subtype, the target metric is optimized on a stratified internal split of the official training set. The held-out testing set does not determine an epoch, model, hyperparameter, or decision threshold.


In [ ]:
# Repository-local path configuration
from pathlib import Path
import os

PROJECT_ROOT = Path(os.environ.get("GEP_PROJECT_ROOT", Path.cwd())).resolve()
DATA_DIR = Path(os.environ.get("GEP_DATA_DIR", PROJECT_ROOT / "data")).resolve()
MODEL_DIR = Path(os.environ.get("GEP_MODEL_DIR", PROJECT_ROOT / "models")).resolve()
RESULTS_DIR = Path(os.environ.get("GEP_RESULTS_DIR", PROJECT_ROOT / "results")).resolve()
FIGURES_DIR = Path(os.environ.get("GEP_FIGURES_DIR", PROJECT_ROOT / "figures")).resolve()

for directory in (MODEL_DIR, RESULTS_DIR, FIGURES_DIR):
    directory.mkdir(parents=True, exist_ok=True)


In [ ]:
# ==== Setup & Imports ====
import os
import random
import numpy as np
import pandas as pd
import sys
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
from threshold_protocol import make_internal_selection_split
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import f1_score, classification_report, confusion_matrix


# ==== Reproducibility ====
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(42)

# ==== Config ====
MODEL_NAME = "bert-base-uncased"
TRAIN_CSV = str(DATA_DIR / 'GEP_train_80_20.csv')

BASE_SAVE_DIR = str(MODEL_DIR / 'BERT_GEP_SUBTYPES_F1')
os.makedirs(BASE_SAVE_DIR, exist_ok=True)

BATCH_SIZE = 16
NUM_EPOCHS = 20
LR = 1e-5
DOC_MAX_TOKENS = 4096
CHUNK_SIZE = 510
MAX_LENGTH = 512
USE_POS_WEIGHT = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ==== Load Data ====
official_train_df = pd.read_csv(TRAIN_CSV)

# ==== Subtypes ====
subtypes = ['Credibility and Obstinacy', 'Compliance', 'Descriptors', 'Misgendering']

# ==== Dataset (Chunking & Pooling Support) ====
class ChunkedTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, chunk_size=510, max_length=512, doc_max_length=4096):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.chunk_size = chunk_size
        self.max_length = max_length
        self.doc_max_length = doc_max_length

    def __len__(self):
        return len(self.texts)

    def chunk_text(self, text):
        token_ids = self.tokenizer.encode(text, add_special_tokens=False, truncation=False)
        token_ids = token_ids[: self.doc_max_length]
        chunks = []
        for i in range(0, len(token_ids), self.chunk_size):
            core = token_ids[i:i + self.chunk_size]
            chunk = [self.tokenizer.cls_token_id] + core + [self.tokenizer.sep_token_id]
            if len(chunk) < self.max_length:
                chunk += [self.tokenizer.pad_token_id] * (self.max_length - len(chunk))
            chunks.append(chunk)
        if not chunks:
            chunk = [self.tokenizer.cls_token_id, self.tokenizer.sep_token_id]
            chunk += [self.tokenizer.pad_token_id] * (self.max_length - len(chunk))
            chunks = [chunk]
        return chunks

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = float(self.labels[idx])
        chunks = self.chunk_text(text)
        return {
            "chunks": torch.tensor(chunks, dtype=torch.long),
            "label": torch.tensor(label, dtype=torch.float),
            "num_chunks": len(chunks)
        }

def bert_collate_fn(batch):
    all_chunks = [item["chunks"] for item in batch]
    labels = torch.tensor([item["label"] for item in batch], dtype=torch.float)
    num_chunks = [item["num_chunks"] for item in batch]
    flat_chunks = torch.cat(all_chunks, dim=0)
    return {"chunks": flat_chunks, "labels": labels, "num_chunks": num_chunks}

# ==== Training & Evaluation Functions ====
def train_one_epoch(model, dataloader, criterion, optimizer, device, scheduler, tokenizer):
    model.train()
    total_loss, total, correct = 0.0, 0, 0
    for batch in tqdm(dataloader, desc="Training", ncols=120):
        chunks = batch["chunks"].to(device)
        labels = batch["labels"].to(device)
        num_chunks = batch["num_chunks"]

        optimizer.zero_grad()
        outputs = model(input_ids=chunks, attention_mask=(chunks != tokenizer.pad_token_id))
        chunk_logits = outputs.logits.squeeze(-1)

        # max-pooling per doc
        pooled_logits = []
        idx = 0
        for nc in num_chunks:
            pooled_logits.append(torch.max(chunk_logits[idx: idx + nc]))
            idx += nc
        pooled_logits = torch.stack(pooled_logits)

        loss = criterion(pooled_logits, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()

        with torch.no_grad():
            probs = torch.sigmoid(pooled_logits)
            preds = (probs >= 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            total_loss += loss.item() * labels.size(0)

    return total_loss / total, 100.0 * correct / total


def evaluate(model, dataloader, criterion, device, tokenizer):
    model.eval()
    total_loss, total, correct = 0.0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Validating", ncols=120):
            chunks = batch["chunks"].to(device)
            labels = batch["labels"].to(device)
            num_chunks = batch["num_chunks"]

            outputs = model(input_ids=chunks, attention_mask=(chunks != tokenizer.pad_token_id))
            chunk_logits = outputs.logits.squeeze(-1)

            pooled_logits = []
            idx = 0
            for nc in num_chunks:
                pooled_logits.append(torch.max(chunk_logits[idx: idx + nc]))
                idx += nc
            pooled_logits = torch.stack(pooled_logits)

            loss = criterion(pooled_logits, labels)
            total_loss += loss.item() * labels.size(0)

            probs = torch.sigmoid(pooled_logits)
            preds = (probs >= 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            all_preds.extend(preds.cpu().numpy().astype(int))
            all_labels.extend(labels.cpu().numpy().astype(int))

    avg_loss = total_loss / total
    avg_acc = 100.0 * correct / total
    avg_f1 = f1_score(all_labels, all_preds, average="macro")
    return avg_loss, avg_acc, avg_f1, np.array(all_preds), np.array(all_labels)

# ==== Sequential Training Across Subtypes ====
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.sep_token
    print("No pad_token_id found; using sep_token as pad_token.")

prev_model_path = MODEL_NAME

for subtype in subtypes:
    print("\n" + "="*80)
    print(f" Training BERT for subtype: {subtype}")
    print("="*80)

    train_df, selection_df = make_internal_selection_split(
        official_train_df, label_column=subtype, selection_fraction=0.15, random_state=42
    )
    train_texts = train_df["text"].astype(str).tolist()
    val_texts = selection_df["text"].astype(str).tolist()
    train_labels = train_df[subtype].astype(int).tolist()
    val_labels = selection_df[subtype].astype(int).tolist()

    # Dataset & DataLoader
    train_dataset = ChunkedTextDataset(train_texts, train_labels, tokenizer)
    val_dataset   = ChunkedTextDataset(val_texts, val_labels, tokenizer)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                              collate_fn=bert_collate_fn, pin_memory=torch.cuda.is_available())
    val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                              collate_fn=bert_collate_fn, pin_memory=torch.cuda.is_available())

    # Load previous model weights to continue sequential fine-tuning
    model = AutoModelForSequenceClassification.from_pretrained(prev_model_path, num_labels=1).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=LR)
    criterion = nn.BCEWithLogitsLoss()
    total_steps = len(train_loader) * NUM_EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, 0, total_steps)

    best_val_f1 = 0.0
    SAVE_DIR = os.path.join(BASE_SAVE_DIR, subtype.replace(" ", "_"))
    os.makedirs(SAVE_DIR, exist_ok=True)

    for epoch in range(1, NUM_EPOCHS + 1):
        print(f"\n===== Epoch {epoch}/{NUM_EPOCHS} for {subtype} =====")
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device, scheduler, tokenizer)
        val_loss, val_acc, val_f1, val_preds, val_labels_np = evaluate(model, val_loader, criterion, device, tokenizer)

        print(f"[{subtype}] Epoch {epoch}: Val Acc={val_acc:.2f}% | Val F1={val_f1:.4f}")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            model.save_pretrained(SAVE_DIR)
            tokenizer.save_pretrained(SAVE_DIR)
            print(f" Saved best {subtype} model (Val F1={best_val_f1:.4f}) → {SAVE_DIR}")

            cm = confusion_matrix(val_labels_np, val_preds)
            print("Confusion Matrix:")
            print(cm)
            print("Classification Report:")
            print(classification_report(val_labels_np, val_preds, digits=3))

    # Continue from the best model of this subtype
    prev_model_path = SAVE_DIR

print("\n All subtype trainings completed successfully (F1-based).")
